# **Prophet ile Satış Tahmin Modeli**

# Giriş

Bu proje, Almanya'da bulunan 1115 farklı Rossmann mağazasından birinin satış verileri aracılığıyla prophet modeli kullanılarak ileriye dönük 6 haftalık günlük satış tahminlemesi üretmek amacıyla geliştirilmiştir.

# Veri Hazırlığı

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import sklearn
from prophet import Prophet

In [ ]:
train = pd.read_csv('train.csv', low_memory=False)
store = pd.read_csv('store.csv')

In [ ]:
train.head(10)

In [ ]:
store.head(10)

In [ ]:
df = pd.merge(train, store, on='Store', how='left')

In [ ]:
df.shape

In [ ]:
df.head(20)

In [ ]:
df["Date"] = pd.to_datetime(df["Date"])
df = df.sort_values("Date")

In [ ]:
best_store = df.groupby('Store')['Sales'].count().idxmax()
df_store = df[df['Store'] == best_store]
print(f"Seçilen Mağaza: {best_store}")
print(f"Toplam kayıt: {len(df_store)}")

Model oluşturmak için en fazla veriye sahip olan mağaza seçildi

In [ ]:
df_store.isnull().sum()

In [ ]:
df_store[['CompetitionDistance' , 'CompetitionOpenSinceMonth' , 'CompetitionOpenSinceYear']]

Mağaza 1'in en yakın rakibi 1270 metre uzaklıkta olup
Eylül 2008'den beri faaliyettedir. Veri setimiz 2013-2015
dönemini kapsadığından rakip mağaza bu süre boyunca sabit
bir konumda bulunmaktadır. Mesafe değeri tüm gözlemler için
değişmediğinden modele dinamik bir katkı sağlamamaktadır.
Bu nedenle CompetitionDistance modele regressor olarak eklenmemelidir.

In [ ]:
df_store["Promo2"].value_counts()

Model oluşturmak için en fazla veriye sahip olan mağaza 1 seçildi ve boş satır sayısı görüntülendi. Promo2SinceWeek , Promo2SinceYear , PromoInterval sütunlarının bütün satırlarının boş olduğu görülüyor. Promo2 sütunu kontrol edildi. Tüm satırları 0 olduğundan mağaza 1'in yıllık/haftalık kampanya yapmadığı görülüyor. Bu yüzden bu sütunlar drop edilebilir, modele etkisi yoktur.

In [ ]:
df_store = df_store.drop(columns=['Promo2' , 'Promo2SinceWeek' , 'Promo2SinceYear' , 'PromoInterval'])

In [ ]:
pd.crosstab(df_store['StateHoliday'], df['Open'])


1 numaralı mağazanın tatil günlerinde kapalı olduğu görülüyor. Satış bulunmadığından dolayı StateHoliday sütunu da drop edilebilir

In [ ]:
df_store = df_store.drop(columns=['StateHoliday'])

In [ ]:
df_store['SchoolHoliday'].value_counts()

In [ ]:
pd.crosstab(df_store['SchoolHoliday'], df['Open'])

Okulların tatil olduğu günlerin çoğunda mağaza açık. Dolayısıyla bunun satış durumuna etkisi olabilir. EDA'de incelenmeli.

In [ ]:
df_store = df_store[df_store['Open'] == 1]
df_store = df_store[df_store['Sales'] > 0]

Mağaza 1'in kapalı olduğu ve dolayısıyla satışın 0 olduğu günler analizden çıkarılmıştır.

In [ ]:
df_store = df_store.drop(columns=['CompetitionDistance', 'CompetitionOpenSinceMonth', 'CompetitionOpenSinceYear'])

Sadece bir mağaza için tahmin geliştirildiğinden drop edilebilir.

# Keşifsel Veri Analizi (EDA)

In [ ]:
df_store.shape

In [ ]:
df_store.dtypes

In [ ]:
df_store.describe()

In [ ]:
plt.figure(figsize=(14,5))
plt.plot(df_store['Date'] , df_store['Sales'])
plt.xlabel('Tarih')
plt.ylabel('Satışlar')
plt.title('Tarih-Satış Grafiği')
plt.show()

In [ ]:
df_store['Yearly'] = df_store['Date'].dt.year
df_store['Monthly'] = df_store['Date'].dt.month
df_store['Daily'] = df_store['Date'].dt.dayofweek

In [ ]:
plt.figure(figsize=(8,5))
df_store.groupby('Yearly')['Sales'].mean().plot(kind='bar')
years = ['2013', '2014', '2015']
plt.xticks(range(3), years, rotation=0)
plt.title('Yıllara Göre Ortalama Satış')
plt.xlabel('Yıllar')
plt.ylabel('Satışlar')
plt.show()

In [ ]:
plt.figure(figsize=(8,5))
df_store.groupby('Monthly')['Sales'].mean().plot(kind='bar')
months = ['Ocak', 'Şubat', 'Mart', 'Nisan', 'Mayıs', 'Haziran', 'Temmuz', 'Ağustos' , 'Eylül' , 'Ekim' , 'Kasım' , 'Aralık']
plt.xticks(range(12), months, rotation=45)
plt.title('Aylara Göre Ortalama Satış')
plt.xlabel('Aylar')
plt.ylabel('Satışlar')
plt.show()

In [ ]:
df_store['Daily'].unique()

In [ ]:
plt.figure(figsize=(8,5))
df_store.groupby('Daily')['Sales'].mean().plot(kind='bar')
days = ['Pazartesi', 'Salı', 'Çarşamba', 'Perşembe', 'Cuma', 'Cumartesi']
plt.xticks(range(6), days, rotation=45)
plt.title('Haftanın Günlerine Göre Ortalama Satış')
plt.xlabel('Günler')
plt.ylabel('Satışlar')
plt.show()

In [ ]:
plt.figure(figsize=(8,5))
df_store.groupby('Promo')['Sales'].mean().plot(kind='bar' , color=['#fc4103' , 'green'])
ticks = ['Kampanya Yok', 'Kampanya Var']
plt.xticks(range(2), ticks, rotation=0)
plt.title('Kampanya Olan vs Olmayan Günlerde Satış')
plt.xlabel('Kampanya Durumu')
plt.ylabel('Satışlar')
plt.show()

In [ ]:
plt.figure(figsize=(8,5))
df_store.groupby('SchoolHoliday')['Sales'].mean().plot(kind='bar', color=['green', 'orange'])
plt.title('Okul Tatili Olan vs Olmayan Günlerde Ortalama Satış')
plt.xticks([0, 1], ['Normal Gün', 'Okul Tatili'], rotation=0)
plt.xlabel('Okul Tatil Durumu')
plt.ylabel('Ortalama Satış')
plt.show()

SchoolHoliday değişkeni incelendiğinde normal günlerle
tatil günleri arasında anlamlı bir satış farkı gözlemlenmemiştir.
Bu nedenle modele regressor olarak eklenmeyecektir.

# Decomposition Analizi

In [ ]:
from statsmodels.tsa.seasonal import seasonal_decompose

df_indexed = df_store.set_index('Date').sort_index()
result = seasonal_decompose(
    df_indexed['Sales'],
    model='additive',    # trend + mevsimsellik + gürültü toplamı
    period=365           # yıllık mevsimsellik (günlük veri olduğu için)
)

result.plot()
plt.suptitle('Mağaza 1 - Satış Ayrıştırması', y=1.02)
plt.tight_layout()
plt.show()

## Decomposition Analizi Yorumu

### Trend
Mağaza 1'in satışları uzun vadede hafif düşüş eğilimindedir.
2013 başında ~4850 seviyesinde olan trend, 2013 ortasında ~4630'a gerilemiş,
2014 ortasında tekrar ~4820'ye çıkmış ve 2014 sonu itibarıyla tekrar düşüşe geçmiştir.

### Mevsimsellik (Seasonality)
Belirgin bir yıllık mevsimsel örüntü gözlemlenmektedir.
Her yıl Ekim-Aralık döneminde satışlarda büyük zirveler oluşmaktadır.
Bu durum Noel alışveriş sezonu ile açıklanabilir.

In [ ]:
mask = (df_store['Date'] >= '2013-08-01') & (df_store['Date'] <= '2013-11-01')
print(df_store[mask][['Date', 'Sales']].head(30))

In [ ]:
mask1 = (df_store['Date'] >= '2014-11-01') & (df_store['Date'] <= '2015-01-01')
print(df_store[mask1][['Date', 'Sales']].head(30))

### Gürültü (Residual)
Residual panelinde gözlemlenen kümelenmeler, seasonal_decompose algoritmasının zaman serisi başlangıç ve bitiş noktalarında yaşadığı edge effect'ten kaynaklanmaktadır. Veri kalitesiyle ilgili bir sorun değildir.

# Model Kurulumu

In [ ]:
test = pd.read_csv('test.csv')
test['Date'] = pd.to_datetime(test['Date'])
print(test['Date'].min(), test['Date'].max())

2015 Temmuz sonrası için 6 haftalık tahmin yapılacak.

In [ ]:
print(df_store.columns.tolist())
print(df_store.shape)
print(df_store.head())

In [ ]:
prophet_df = df_store[['Date', 'Sales', 'Promo']].rename(
    columns={'Date': 'ds', 'Sales': 'y'}
)

print(prophet_df.shape)
print(prophet_df.head())

In [ ]:
train_df = prophet_df[prophet_df['ds'] < '2015-07-01']
test_df = prophet_df[prophet_df['ds'] >= '2015-07-01']

print(f"Train: {len(train_df)} kayıt")
print(f"Test: {len(test_df)} kayıt")

In [ ]:
model = Prophet(
    yearly_seasonality=True,
    weekly_seasonality=True,
    daily_seasonality=False,
    seasonality_mode='multiplicative'
)

model.add_regressor('Promo')

model.fit(train_df)

In [ ]:
from prophet.diagnostics import cross_validation, performance_metrics

df_cv = cross_validation(
    model,
    initial='500 days', period='90 days', horizon='30 days')

df_p = performance_metrics(df_cv)
print(df_p[['horizon', 'mae', 'rmse', 'mape']].head(30))

### Cross Validation Sonuçları

Model, 2014-2015 dönemi boyunca 5 farklı zaman diliminde test edilmiştir.
30 günlük tahmin ufkunda MAPE değerleri genel olarak %8-10 bandında seyretmiştir.

Ancak 30 günlük cross validation çıktısı incelendiğinde belirgin bir örüntü
dikkat çekmektedir: 7. günde %13.8, 14. günde %14.1, 21. günde %21.3 ile
hafta geçiş noktalarında MAPE değerleri sistematik olarak yükselmektedir.
Bu durum modelin haftalık mevsimsellik sınırlarında kısa vadeli
dalgalanmaları tam olarak yakalayamadığına işaret etmektedir.

Hafta geçişlerinin ardından MAPE tekrar normale döndüğünden bu sınırlılık
modelin genel güvenilirliğini bozmamaktadır.

In [ ]:
metrics = {'Metrik': ['MAE', 'RMSE', 'MAPE'],
           'Değer': [318.43, 406.85, '7.12%']}
pd.DataFrame(metrics)

In [ ]:
future = model.make_future_dataframe(periods=48, freq='D')
# Train'deki Promo değerleri alındı
future = future.merge(
    prophet_df[['ds', 'Promo']],
    on='ds',
    how='left'
)

# Gelecek günler için Promo bilinmiyor o yüzden 0 kabul edildi.
future['Promo'] = future['Promo'].fillna(0)

In [ ]:
forecast = model.predict(future)

In [ ]:
print(forecast[['ds', 'yhat', 'yhat_lower', 'yhat_upper']].tail(20))

In [ ]:
forecast_test = forecast[forecast['ds'] >= '2015-07-01'][['ds', 'yhat', 'yhat_lower', 'yhat_upper']]

comparison = test_df.merge(forecast_test, on='ds', how='left')
print(comparison.head(10))

In [ ]:
from sklearn.metrics import mean_absolute_error, mean_squared_error

mae = mean_absolute_error(comparison['y'], comparison['yhat'])
rmse = np.sqrt(mean_squared_error(comparison['y'], comparison['yhat']))
mape = np.mean(np.abs((comparison['y'] - comparison['yhat']) / comparison['y'])) * 100

print(f"MAE:  {mae:.2f}")
print(f"RMSE: {rmse:.2f}")
print(f"MAPE: {mape:.2f}%")

##Model Performansı
Model, gerçek satış değerlerinden ortalama %7.12 sapma göstermiştir. Perakende satış tahmininde %10 altı mükemmel kabul edildiğinden
model başarılı bir performans sergilemiştir.

In [ ]:
plt.figure(figsize=(14, 6))
plt.plot(comparison['ds'], comparison['y'],
         label='Gerçek Satışlar', color='blue')
plt.plot(comparison['ds'], comparison['yhat'],
         label='Tahmin', color='red', linestyle='--')
plt.fill_between(comparison['ds'],
                 comparison['yhat_lower'],
                 comparison['yhat_upper'],
                 alpha=0.3, color='orange', label='Güven Aralığı')
plt.title('Mağaza 1 - Gerçek vs Tahmin Satışlar')
plt.xlabel('Tarih')
plt.ylabel('Satış')
plt.legend()
plt.show()

## Gerçek vs Tahmin Grafiği Yorumu

Model, Temmuz 2015 dönemindeki gerçek satışları başarıyla tahmin etmiştir.
Haftalık mevsimsellik örüntüleri doğru yakalanmış, gerçek satış değerlerinin
büyük çoğunluğu güven aralığı içinde kalmıştır. %7.12 MAPE değeri grafikte
de görsel olarak doğrulanmaktadır.

In [ ]:
model.plot_components(forecast)
plt.show()

## Model Bileşenleri Yorumu

- **Trend:** Satışlar 2013'te düşüş göstermiş, 2014 sonrası stabilleşmiştir.
- **Haftalık:** Cumartesi en yüksek (%17), Perşembe en düşük (%-10) satış günüdür.
- **Yıllık:** Aralık ayında Noel sezonu nedeniyle %50'ye yakın artış gözlemlenmektedir.
- **Promosyon:** Promosyon günlerinde satışlar %30 artmaktadır.

#  Genel Rapor

## Proje Amacı
Rossmann mağaza zincirinin geçmiş satış verileri kullanılarak
Meta Prophet modeli ile gelecek dönem satış tahmini yapılması
ve modelin performansının değerlendirilmesi.

## Veri Seti
- **Kaynak:** Rossmann Store Sales (Kaggle)
- **Kapsam:** Mağaza 1, 2013-2015 dönemi
- **Toplam Kayıt:** 781 gün (kapalı günler çıkarıldıktan sonra)
- **Train/Test:** 754 / 27 kayıt

## Veri Hazırlığı
- Kapalı mağaza günleri (Open=0) ve sıfır satış kayıtları veri setinden çıkarıldı
- Tatil günlerinde (StateHoliday) mağazanın kapalı olduğu tespit edildi
- SchoolHoliday değişkeninin satışlar üzerinde anlamlı etkisi gözlemlenmedi
- Rakip mağaza mesafesi (1270m) tüm dönem boyunca sabit kaldığından
  modele eklenmedi

## Model
- **Algoritma:** Prophet
- **Mevsimsellik:** Yıllık + Haftalık
- **Regressor:** Promo (promosyon günleri)
- **Mod:** Multiplicative

## Performans Sonuçları

| Metrik | Değer |
|--------|-------|
| MAE    | 318.43 |
| RMSE   | 406.85 |
| MAPE   | %7.12  |

## Temel Bulgular
- Mağaza satışları uzun vadede hafif düşüş trendindedir
- Cumartesi en yüksek, Perşembe en düşük satış günüdür
- Aralık ayında Noel sezonu nedeniyle %50'ye yakın satış artışı gözlemlenmektedir
- Promosyon günlerinde satışlar ortalama %30 artmaktadır
- Model gerçek satışlardan ortalama %7.12 sapma göstermiş olup
  perakende sektöründe mükemmel kabul edilen %10 eşiğinin altındadır

## Sonuç
Prophet modeli Rossmann Mağaza 1'in satış örüntülerini başarıyla
öğrenmiş ve gelecek dönem tahminlerini yüksek doğrulukla üretmiştir.
Haftalık ve yıllık mevsimsellik bileşenleri net biçimde yakalanmış,
promosyon etkisi modele başarıyla entegre edilmiştir.